# 03 — NLP Pathology Label Imputation

Regex + ConText negation detection to impute missing `Pathology.Categories` from free-text notes.

In [1]:
import pandas as pd
from gtex_biomarkers.config import Config
from gtex_biomarkers.data import load_cache
from gtex_biomarkers.labels import extract_categories, CATEGORY_PATTERNS

Config.ensure_dirs()

# Load cached data (run notebook 01 first)
_, _, _, df_meta_url, _ = load_cache()

Loaded cache from /Users/rsinha/Library/CloudStorage/OneDrive-SanfordBurnhamPrebysMedicalDiscoveryInstitute/Desktop/gtex_gene_expression/data/cache/processed_data.pkl
  X_wb: 803 samples × 59033 genes


## Liver Pathology — Most Common Raw Strings

In [2]:
liver_all = df_meta_url[df_meta_url["Tissue"].astype(str) == "Liver"].copy()
liver_all["Pathology.Categories"].fillna("NA").value_counts()

Pathology.Categories
steatosis                                                    137
congestion                                                   137
NA                                                           101
congestion, steatosis                                         66
cirrhosis, steatosis                                          13
                                                            ... 
hemorrhage, steatosis                                          1
cirrhosis, fibrosis, inflammation                              1
cirrhosis, fibrosis, hyalinization, steatosis                  1
cirrhosis, hyalinization, inflammation, necrosis, pigment      1
cirrhosis, inflammation, steatosis                             1
Name: count, Length: 66, dtype: int64

## Validate on Ground-Truth Rows

In [3]:
# Apply extraction to rows where categories already exist
liver_gt = liver_all[liver_all["Pathology.Categories"].notna()].copy()
liver_gt["pred_list"] = liver_gt["Pathology.Notes"].apply(extract_categories)

print(f"Ground-truth rows: {len(liver_gt)}")
display(liver_gt[["Pathology.Categories", "Pathology.Notes", "pred_list"]].head(10))

Ground-truth rows: 509


,Pathology.Categories,Pathology.Notes,pred_list
10,necrosis,"2 pieces, subtotal massive hepatic necrosis",[necrosis]
66,congestion,"2 pieces, congestion",[congestion]
94,"fibrosis, inflammation",2 pieces; chronic inflammation and fibrosis,"[fibrosis, inflammation]"
202,steatosis,2 pieces; mild macrovesicular steatosis,[steatosis]
236,"fibrosis, inflammation, steatosis","2 pieces, mild steatosis, widening of sinusoid...","[fibrosis, inflammation, steatosis]"
264,"congestion, fibrosis, inflammation, steatosis","2 pieces, passive congestion, bridging fibrosi...","[congestion, fibrosis, inflammation, steatosis]"
321,congestion,"2 pieces, prominent hepatic congestion/'nutmeg...",[congestion]
355,"cirrhosis, steatosis","2 pieces diffuse established cirrhosis, and ma...","[cirrhosis, steatosis]"
421,congestion,"2 pieces, congestion",[congestion]
449,necrosis,"2 pieces, foci of hepatocyte necrosis and ball...",[necrosis]


## Apply to Missing Rows & Save

In [4]:
liver_missing = liver_all[liver_all["Pathology.Categories"].isna()].copy()
print(f"Rows with missing categories: {len(liver_missing)}")

liver_missing["pred_list"] = liver_missing["Pathology.Notes"].apply(extract_categories)
liver_missing["Pathology.Categories.Imputed"] = liver_missing["pred_list"].apply(
    lambda lst: ", ".join(lst) if lst else float("nan")
)

filled = liver_missing["Pathology.Categories.Imputed"].notna().sum()
print(f"Imputed: {filled} / {len(liver_missing)}")

# Combine original + imputed
liver_all["Pathology.Categories.Final"] = liver_all["Pathology.Categories"].copy()
imputed_mask = liver_all["Pathology.Categories"].isna()
liver_all.loc[imputed_mask, "Pathology.Categories.Final"] = (
    liver_missing["Pathology.Categories.Imputed"]
)

liver_all["SUBJID"] = liver_all["Tissue.Sample.ID"].astype(str).str.split("-").str[:2].str.join("-")
liver_all.to_csv(Config.PROCESSED_DIR / "liver_pathology_labels_imputed.csv", index=False)
print(f"Saved → {Config.PROCESSED_DIR / 'liver_pathology_labels_imputed.csv'}")

Rows with missing categories: 101
Imputed: 15 / 101
Saved → /Users/rsinha/Library/CloudStorage/OneDrive-SanfordBurnhamPrebysMedicalDiscoveryInstitute/Desktop/gtex_gene_expression/data/processed/liver_pathology_labels_imputed.csv
